# Scenario 17: Cost Tracking — Pure OTel + Custom Scorer

Third of three cost-tracking demos split out from `14_cost_tracking.ipynb`.

**Focus:** the **pure-OTel** ingest path (`OTLPSpanExporter` + manual spans, exactly like `12_manual_otel.ipynb`), with cost tracking and a custom LLM-as-judge scorer **bound to the OTel log stream** so `metric_cost` is populated for OTel-sourced traces.

## What this notebook proves (corrected vs. earlier doc)

The original `14_cost_tracking.ipynb` Step 10 left two open questions for the OTel path. Reading the actual ingest code (`api/api/services/otel_v2/components/processor.py:60-66`) shows that an earlier version of the Q&A doc was **wrong** about how OTel cost works. The corrected questions this notebook answers:

1. **Does the runners re-price OTel-sourced tokens?** **Yes.** OTel ingest normalizes `gen_ai.request.model` → `model`, `gen_ai.usage.input_tokens`/`output_tokens` → `input_tokens`/`output_tokens` on the internal chain/response rows, then the same `runners/utils/costs/cost.py:calculate_cost()` runs on those rows. Cohort A (tokens-only, no cost attribute) verifies this.
2. **Is the explicit `cost` / `gen_ai.usage.cost` span attribute consumed?** **No.** A search of `api/api/services/otel_v2/` finds no consumer for either name. Cohort B verifies the precomputed value is *ignored* — runners re-prices regardless. (This is the interesting demo finding: customers who want a custom unit cost should configure a per-org `model_price_overrides` row, not bake it into the span.)
3. **Do scorers fire on OTel traces, and does `metric_cost` get populated?** Yes, **but only if `enable_metrics()` is called for the OTel log stream**. Step 4 binds the custom scorer; without it, scorers don't fire and `metric_cost` stays `absent` for OTel-sourced traces.

## Compare to siblings

| Aspect | 12_manual_otel | 15_cost_native | 16_cost_custom_judge | 17 (this) |
|---|---|---|---|---|
| Trace path | Pure OTel | Galileo SDK | Galileo SDK | **Pure OTel** |
| Scorer | None | `correctness` | `custom_LLM_boolean_metric` | **`custom_LLM_boolean_metric`** |
| Cost demo | No | Yes | Yes | **Yes (via OTel)** |

## Step 0: Load environment variables

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

GALILEO_SDK_SOURCE = None
for root in [Path.cwd(), *Path.cwd().parents]:
    local_sdk_src = root / 'galileo-python' / 'src'
    if (local_sdk_src / 'galileo').exists():
        GALILEO_SDK_SOURCE = local_sdk_src
        if str(local_sdk_src) not in sys.path:
            sys.path.insert(0, str(local_sdk_src))
        break

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

for candidate in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

Loaded credentials from /Users/binliu/Documents/rungalileo/demos/galileo-test/.env


## Step 0b: Imports and constants

Nothing in the trace path imports Galileo SDK helpers — only OTel + raw `openai`. Galileo imports here are for project setup, scorer creation, and post-hoc REST queries (which is fine; they don't affect span emission).

In [2]:
import json
import os
import time
from urllib.parse import urljoin, quote

import openai as raw_openai

# Galileo (used only for project + scorer setup, not in span path)
from galileo import galileo_context
from galileo.config import GalileoPythonConfig
from galileo.log_streams import create_log_stream, enable_metrics, get_log_stream
from galileo.metrics import create_custom_llm_metric
from galileo.projects import delete_project
from galileo.resources.api.trace import query_traces_projects_project_id_traces_search_post
from galileo.resources.models.log_records_query_request import LogRecordsQueryRequest
from galileo.resources.models.output_type_enum import OutputTypeEnum
from galileo.scorers import Scorers
from galileo_core.constants.request_method import RequestMethod
from galileo_core.schemas.logging.step import StepType

# OpenTelemetry (the actual trace path)
from opentelemetry import trace as otel_trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

PROJECT_NAME = 'GalileoEval_S17_CostOTel'
LOG_STREAM_NAME = 'cost-otel'
MODEL = 'gpt-4o-mini'
CUSTOM_JUDGE_MODEL = 'md0033_openai_gpt4.1miniptu'  # custom-integrated judge LLM
CUSTOM_METRIC_NAME = 'custom_LLM_boolean_metric_s17'

print(f'Galileo SDK source: {GALILEO_SDK_SOURCE or "installed package"}')
PROJECT_NAME, LOG_STREAM_NAME, MODEL, CUSTOM_JUDGE_MODEL, CUSTOM_METRIC_NAME

Galileo SDK source: /Users/binliu/Documents/rungalileo/galileo-python/src


/Users/binliu/Documents/rungalileo/demos/galileo-test/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('GalileoEval_S17_CostOTel',
 'cost-otel',
 'gpt-4o-mini',
 'md0033_openai_gpt4.1miniptu',
 'custom_LLM_boolean_metric_s17')

## Step 1: Initialize Galileo project + OTel log stream

We use `galileo_context.init()` only to ensure the project + log stream exist and to read back `project_id` / `log_stream_id` for later REST queries. After this, the SDK is not in the span path — Step 2's pipeline emits OTel spans directly.

In [3]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

ls = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME) or \
     create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger_inst = galileo_context.get_logger_instance()
project_id = getattr(logger_inst, 'project_id', None)
log_stream_id = getattr(logger_inst, 'log_stream_id', None)

GALILEO_API_URL = str(config.api_url)
GALILEO_API_KEY = config.api_key.get_secret_value() if config.api_key else None
OTEL_TRACES_ENDPOINT = os.environ.get(
    'OTEL_EXPORTER_OTLP_TRACES_ENDPOINT',
    urljoin(GALILEO_API_URL if GALILEO_API_URL.endswith('/') else GALILEO_API_URL + '/', 'otel/traces'),
)

console_base = str(config.console_url)
log_stream_url = f'{console_base}project/{project_id}/log-streams/{log_stream_id}'
trends_url = f'{log_stream_url}?tab=trends'

print('Log stream    :', log_stream_url)
print('Trends tab    :', trends_url)
print('OTLP endpoint :', OTEL_TRACES_ENDPOINT)

Log stream    : https://console-bin-citizens.gcp-dev.galileo.ai/project/a198016f-83d6-4d69-983f-391cb7c8ba8e/log-streams/5fa91e4f-ed10-442d-bbe8-97899bf30f38
Trends tab    : https://console-bin-citizens.gcp-dev.galileo.ai/project/a198016f-83d6-4d69-983f-391cb7c8ba8e/log-streams/5fa91e4f-ed10-442d-bbe8-97899bf30f38?tab=trends
OTLP endpoint : https://api-bin-citizens.gcp-dev.galileo.ai/otel/traces


## Step 2: Build the pure OTel pipeline

Verbatim copy of the customer-style setup from `12_manual_otel.ipynb`: `TracerProvider` + `Resource` + raw `OTLPSpanExporter` with the three Galileo routing headers (`Galileo-API-Key`, `project`, `logstream`) + `BatchSpanProcessor`. Zero OpenInference, zero Galileo SDK helpers in the span path.

In [4]:
resource = Resource.create({
    'service.name': 'galileo-cost-otel-demo',
    'service.version': '1.0.0',
    'deployment.environment': 'notebook',
})

# Re-runnable: tear down any prior provider before installing a new one.
try:
    otel_trace.get_tracer_provider().shutdown()
except Exception:
    pass

tracer_provider = TracerProvider(resource=resource)
exporter = OTLPSpanExporter(
    endpoint=OTEL_TRACES_ENDPOINT,
    headers={
        'Galileo-API-Key': GALILEO_API_KEY,
        'project': PROJECT_NAME,
        'logstream': LOG_STREAM_NAME,
    },
)
span_processor = BatchSpanProcessor(exporter)
tracer_provider.add_span_processor(span_processor)
otel_trace.set_tracer_provider(tracer_provider)

tracer = otel_trace.get_tracer('cost-otel')
raw_client = raw_openai.OpenAI()  # plain OpenAI client -- NOT galileo.openai

print('Pure OTel pipeline ready (no OpenInference, no Galileo SDK helpers in span path):')
print(f'  Endpoint -> {OTEL_TRACES_ENDPOINT}')
print(f'  Routing  -> project={PROJECT_NAME}, logstream={LOG_STREAM_NAME}')

Pure OTel pipeline ready (no OpenInference, no Galileo SDK helpers in span path):
  Endpoint -> https://api-bin-citizens.gcp-dev.galileo.ai/otel/traces
  Routing  -> project=GalileoEval_S17_CostOTel, logstream=cost-otel


## Step 3: Verify the custom judge model is priced

Same check as Notebook 16 — without a price override, judge calls run but `metric_cost` stays `None`.

In [5]:
judge_prices = config.api_client.request(
    method=RequestMethod.GET,
    path=f'/models/{quote(CUSTOM_JUDGE_MODEL, safe="")}/prices',
)
print(f'--- Prices for {CUSTOM_JUDGE_MODEL} ---')
print(json.dumps(judge_prices, indent=2, default=str))

if not judge_prices.get('override'):
    print(f'\n[WARN] No price override for {CUSTOM_JUDGE_MODEL}. metric_cost will be None.')
else:
    print('\nOK -- override present.')

--- Prices for md0033_openai_gpt4.1miniptu ---
{
  "override": {
    "input_price": 10000.0,
    "output_price": 100000.0
  }
}

OK -- override present.


## Step 4: Bind `custom_LLM_boolean_metric` to the OTel log stream

**This is the step the original notebook missed.** `enable_metrics()` is per-stream, and the OTel log stream had no scorers attached, so `metric_cost` always came back `None`.

We create the scorer (or reuse it if it exists) and bind it to *this* log stream so the runners' Celery scoring job will pick up OTel-sourced traces.

In [6]:
existing = Scorers().list(name=CUSTOM_METRIC_NAME)
if existing:
    print(f'Reusing metric: {CUSTOM_METRIC_NAME} (scorer_id={existing[0].id})')
else:
    create_custom_llm_metric(
        name=CUSTOM_METRIC_NAME,
        user_prompt=(
            'You are evaluating a customer support chatbot reply.\n'
            'The reply is GOOD if BOTH:\n'
            '  (a) It is at most 2 sentences long.\n'
            '  (b) It does not fabricate specifics (no made-up order IDs, prices, or links).\n'
            'Return true if the reply is GOOD, false otherwise.'
        ),
        node_level=StepType.llm,
        output_type=OutputTypeEnum.BOOLEAN,
        cot_enabled=True,
        model_name=CUSTOM_JUDGE_MODEL,
        num_judges=3,
        description=f'Boolean LLM-as-judge running on {CUSTOM_JUDGE_MODEL} for OTel cohort',
        tags=['demo', 'cost-tracking', 'otel'],
    )
    print(f'Created metric: {CUSTOM_METRIC_NAME}')

enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[CUSTOM_METRIC_NAME],
)
print(f'Metrics enabled on {LOG_STREAM_NAME}: [{CUSTOM_METRIC_NAME}]')

Reusing metric: custom_LLM_boolean_metric_s17 (scorer_id=2938dd9d-f931-45a2-9858-a3706dc7a638)
Metrics enabled on cost-otel: [custom_LLM_boolean_metric_s17]


## Step 5: Send two OTel cohorts (with synthetic reasoning tokens)

- **Cohort A — tokens-only**: sets `gen_ai.usage.input_tokens`/`output_tokens` + `gen_ai.request.model`. No cost attribute. Tests whether the runners re-prices OTel tokens by model name.
- **Cohort B — with-cost**: same as A plus precomputed `gen_ai.usage.cost` and `cost` span attributes. Tests whether either attribute path is consumed by ingest.

Each span also carries:
- `cost.cohort=...` so we can tell the records apart in Step 6 (note: this typically does NOT survive ingest filtering — see Step 6's note).
- **`gen_ai.usage.details.reasoning_tokens`** — synthetic value. The OTel processor at `api/api/services/otel_v2/components/processor.py:65, 754` reads this attribute verbatim and writes it as `num_reasoning_tokens` on the trace metrics. Since `gpt-4o-mini` doesn't actually emit reasoning tokens, we set a synthetic value (`out_t // 2 + 7`) to prove the OTel attribute path is what populates the metric — the value is whatever the SDK chose to put there, no transformation, no model-name check.

`_usage_in_out` handles both old (`prompt_tokens`/`completion_tokens`) and new (`input_tokens`/`output_tokens`) openai SDK schemas.

In [7]:
GPT_4O_MINI_IN_PER_M = 0.15
GPT_4O_MINI_OUT_PER_M = 0.60

def _usage_in_out(usage):
    if usage is None:
        return 0, 0
    in_t = getattr(usage, 'input_tokens', None) or getattr(usage, 'prompt_tokens', None) or 0
    out_t = getattr(usage, 'output_tokens', None) or getattr(usage, 'completion_tokens', None) or 0
    return in_t, out_t

def precompute_cost(in_t, out_t):
    return (in_t * GPT_4O_MINI_IN_PER_M + out_t * GPT_4O_MINI_OUT_PER_M) / 1_000_000

def synthetic_reasoning_tokens(out_t):
    """Deterministic synthetic value so the verdict can confirm exact propagation.
    Real reasoning tokens would only come from o-series models; we're using this to
    prove the OTel attribute path is what populates `num_reasoning_tokens`.
    """
    return out_t // 2 + 7

def otel_chat(prompt, cohort, with_cost=False):
    messages = [
        {'role': 'system', 'content': 'You are a concise customer support agent. Answer in 1-2 sentences.'},
        {'role': 'user', 'content': prompt},
    ]
    with tracer.start_as_current_span(
        'chat',
        attributes={
            'gen_ai.system': 'openai',
            'gen_ai.operation.name': 'chat',
            'gen_ai.request.model': MODEL,
            'gen_ai.input.messages': json.dumps(messages),
            'cost.cohort': cohort,
        },
    ) as span:
        response = raw_client.chat.completions.create(
            model=MODEL, messages=messages, max_tokens=80,
        )
        answer = response.choices[0].message.content
        in_t, out_t = _usage_in_out(response.usage)
        span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'assistant', 'content': answer}]))
        span.set_attribute('gen_ai.response.model', response.model)
        span.set_attribute('gen_ai.usage.input_tokens', in_t)
        span.set_attribute('gen_ai.usage.output_tokens', out_t)
        # Synthetic reasoning tokens — see Step 5 markdown for why
        synth_reasoning = synthetic_reasoning_tokens(out_t)
        span.set_attribute('gen_ai.usage.details.reasoning_tokens', synth_reasoning)
        if with_cost:
            precomputed = precompute_cost(in_t, out_t)
            span.set_attribute('gen_ai.usage.cost', precomputed)
            span.set_attribute('cost', precomputed)
            return in_t, out_t, precomputed, synth_reasoning
        return in_t, out_t, None, synth_reasoning

prompts_a = [
    'How do I reset my password?',
    'What are your business hours?',
    'Can I get a refund on my last order?',
]
prompts_b = [
    'Where is my package right now?',
    'How do I contact a human agent?',
    'Is there a mobile app I can download?',
]

# Track expected reasoning_tokens per prompt so Step 6 can verify exact propagation.
expected_reasoning = {}

print('--- Cohort A (tokens-only) ---')
for p in prompts_a:
    in_t, out_t, _, synth = otel_chat(p, cohort='tokens-only', with_cost=False)
    expected_reasoning[p] = synth
    print(f'  {in_t:>4} in / {out_t:>4} out  | reasoning(synthetic)={synth}  |  {p}')

print('\n--- Cohort B (with explicit cost attribute) ---')
for p in prompts_b:
    in_t, out_t, c, synth = otel_chat(p, cohort='with-cost', with_cost=True)
    expected_reasoning[p] = synth
    print(f'  {in_t:>4} in / {out_t:>4} out  | precomputed=${c:.6f}  | reasoning(synthetic)={synth}  |  {p}')

span_processor.force_flush(40000)
print(f'\n-> 6 OTel traces sent and flushed ({len(prompts_a)} cohort A, {len(prompts_b)} cohort B).')

--- Cohort A (tokens-only) ---
    34 in /   34 out  | reasoning(synthetic)=24  |  How do I reset my password?
    33 in /   16 out  | reasoning(synthetic)=15  |  What are your business hours?
    37 in /   18 out  | reasoning(synthetic)=16  |  Can I get a refund on my last order?

--- Cohort B (with explicit cost attribute) ---
    34 in /   14 out  | precomputed=$0.000013  | reasoning(synthetic)=14  |  Where is my package right now?
    35 in /   22 out  | precomputed=$0.000018  | reasoning(synthetic)=18  |  How do I contact a human agent?
    36 in /   30 out  | precomputed=$0.000023  | reasoning(synthetic)=22  |  Is there a mobile app I can download?

-> 6 OTel traces sent and flushed (3 cohort A, 3 cohort B).


## Step 6: Poll back and verify cost + metric_cost (trace-level)

We wait up to 3 minutes — OTel ingest, runners cost job, and scorer Celery job all need to drain.

Three things to look for in the trace-level response:
1. **Token counts** populated → OTel ingest mapped `gen_ai.usage.*` attributes correctly (`processor.py:60-66`).
2. **`cost`** populated for **all records** → runners re-priced from tokens. Cohort B's explicit `gen_ai.usage.cost` span attribute is *not* consulted; the value runners writes will match Cohort A's pricing logic.
3. **`metric_cost`** state — same asymmetry as notebook 16: with a custom-integrated judge priced only via `model_price_overrides`, `Model.cost()` doesn't read that override and the value comes back populated as `$0`.

`num_reasoning_tokens` is verified separately in **Step 6b** below — it's a span-level metric and does not roll up to the trace.

**Note on cohort detection:** the spans set `cost.cohort=tokens-only` / `with-cost` as a custom attribute. OTel ingest does *not* surface unrecognized custom span attributes onto `record.user_metadata` or `record.metadata` — only the recognized `gen_ai.*` set does. So `cohort_of()` returns `unknown` for every record, and the verdict aggregates totals across the run rather than per-cohort.

In [8]:
def fetch_traces(target_log_stream_id, limit=50):
    req = LogRecordsQueryRequest(log_stream_id=target_log_stream_id, limit=limit)
    return query_traces_projects_project_id_traces_search_post.sync(
        project_id=project_id, client=config.api_client, body=req,
    )

def _props(obj, attr):
    target = getattr(obj, attr, None)
    if target is None or not hasattr(target, 'additional_properties'):
        return {}
    return target.additional_properties or {}

def extract_user_cost(record):
    info = _props(record, 'metric_info')
    cost_metric = info.get('cost')
    return getattr(cost_metric, 'value', None) if cost_metric else None

def extract_scorer_metric_cost(record, scorer_name):
    """Return (state, value). state in {'populated','null','absent'}."""
    info = _props(record, 'metric_info')
    entry = info.get(scorer_name)
    if entry is None:
        return 'absent', None
    cost = getattr(entry, 'cost', None)
    if isinstance(cost, (int, float)):
        return 'populated', float(cost)
    return 'null', None

def extract_tokens(record):
    """Trace-level metrics roll up num_input/output/total_tokens via `additional_properties`."""
    metrics_obj = getattr(record, 'metrics', None)
    if metrics_obj is None:
        return 0, 0
    ap = _props(record, 'metrics')
    in_t = ap.get('num_input_tokens') or getattr(metrics_obj, 'num_input_tokens', None) or 0
    out_t = ap.get('num_output_tokens') or getattr(metrics_obj, 'num_output_tokens', None) or 0
    return in_t, out_t

def cohort_of(record):
    for attr in ('user_metadata', 'metadata'):
        props = _props(record, attr)
        for key in ('cost.cohort', 'cost_cohort'):
            if key in props:
                return props[key]
    return 'unknown'

deadline = time.time() + 180
records = []
while time.time() < deadline:
    resp = fetch_traces(log_stream_id)
    records = getattr(resp, 'records', []) or []
    have_cost = any(extract_user_cost(r) is not None for r in records)
    if records and have_cost:
        break
    time.sleep(10)

print(f'OTel records returned: {len(records)}\n')
print(f'{"cohort":<14} {"in/out":>10}  {"cost":>14}  {"expected":>14}  {"metric_cost":>14}  prompt')
print('-' * 110)

overall = {
    'cost_populated': 0, 'cost_total': 0.0, 'expected_total': 0.0,
    'metric_states': {'populated': 0, 'null': 0, 'absent': 0},
    'metric_total': 0.0,
}
for r in records[:20]:
    cohort = cohort_of(r)
    in_t, out_t = extract_tokens(r)
    cost = extract_user_cost(r)
    state, metric_cost = extract_scorer_metric_cost(r, CUSTOM_METRIC_NAME)
    expected = precompute_cost(in_t, out_t)
    raw = getattr(r, 'input_', '') or ''
    prompt = (raw if isinstance(raw, str) else '')[:30]
    if isinstance(cost, (int, float)):
        overall['cost_populated'] += 1
        overall['cost_total'] += cost
        overall['expected_total'] += expected
    overall['metric_states'][state] += 1
    if state == 'populated':
        overall['metric_total'] += metric_cost
    metric_disp = f'${metric_cost:.6f}' if state == 'populated' else state
    print(
        f'{cohort:<14} {int(in_t):>4}/{int(out_t):>4}'
        f'  {(f"${cost:.6f}" if isinstance(cost,(int,float)) else "None"):>14}'
        f'  {f"${expected:.6f}":>14}'
        f'  {metric_disp:>14}'
        f'  {prompt}'
    )

print('\n--- Verdict (trace-level) ---')
if not records:
    print('  No OTel records yet. Re-run after another minute.')
else:
    n = len(records)
    ms = overall['metric_states']
    print(
        f'  user cost              : populated={overall["cost_populated"]}/{n}'
        f'  sum=${overall["cost_total"]:.6f}'
        f'  (expected ${overall["expected_total"]:.6f}; difference ${overall["cost_total"] - overall["expected_total"]:+.6f})'
    )
    print(
        f'  metric_cost            : populated={ms["populated"]}, null={ms["null"]}, absent={ms["absent"]}'
        f'  sum=${overall["metric_total"]:.6f}'
    )
    if overall['cost_populated'] == n and abs(overall['cost_total'] - overall['expected_total']) < 1e-9:
        print(
            '\n  PROVEN (cost): runners re-priced every OTel-sourced record from gen_ai.usage.* tokens.\n'
            '  Cohort B\'s precomputed gen_ai.usage.cost / cost span attributes were IGNORED; the\n'
            '  values written are identical to the costs.json formula applied to the token counts.'
        )
    if ms['populated'] > 0 and overall['metric_total'] == 0.0:
        print(
            '\n  metric_cost = $0 is the same asymmetry that notebook 16 documented:\n'
            '  Model.cost() (orbit/.../schemas/model.py:155-208) does not read the per-org\n'
            '  model_price_overrides table — only costs.json. Custom-integrated judges priced\n'
            '  via overrides therefore produce metric_cost = 0.'
        )

OTel records returned: 6

cohort             in/out            cost        expected     metric_cost  prompt
--------------------------------------------------------------------------------------------------------------
unknown          36/  30       $0.000023       $0.000023            null  Is there a mobile app I can do
unknown          35/  22       $0.000018       $0.000018      $13.600000  How do I contact a human agent
unknown          34/  14       $0.000014       $0.000013            null  Where is my package right now?
unknown          37/  18       $0.000016       $0.000016            null  Can I get a refund on my last 
unknown          33/  16       $0.000015       $0.000015      $14.960000  What are your business hours?
unknown          34/  34       $0.000025       $0.000025      $14.740000  How do I reset my password?

--- Verdict (trace-level) ---
  user cost              : populated=6/6  sum=$0.000112  (expected $0.000112; difference $+0.000000)
  metric_cost          

## Step 6b — §1.7: Verify `num_reasoning_tokens` via the **spans** search API

`num_reasoning_tokens` is recorded **per LLM span** by the OTel processor at `processor.py:746-754`. It is *not* aggregated up to the trace, so reading from the trace search response will return `None`. To verify the synthetic value Step 5 placed on the OTel attribute `gen_ai.usage.details.reasoning_tokens`, we query spans directly with `query_spans_projects_project_id_spans_search_post` and read `span.metrics.additional_properties['num_reasoning_tokens']`.

If the values match our synthetic formula `out_t // 2 + 7` for every LLM span, the documented OTel pipeline (attribute → processor → ClickHouse → spans-search response) is verbatim end-to-end.

In [9]:
from galileo.resources.api.trace import query_spans_projects_project_id_spans_search_post
from galileo.resources.models.log_records_sort_clause import LogRecordsSortClause

# Sort by created_at descending so we look at THIS run's spans, not stale ones from
# earlier runs that might pre-date the Step 5 OTel attribute setting.
sort_desc = LogRecordsSortClause(column_id='created_at', ascending=False)
span_resp = query_spans_projects_project_id_spans_search_post.sync(
    project_id=project_id, client=config.api_client,
    body=LogRecordsQueryRequest(log_stream_id=log_stream_id, limit=50, sort=sort_desc),
)
span_records = getattr(span_resp, 'records', []) or []
llm_spans = [s for s in span_records if getattr(s, 'type_', None) == 'llm']

# Verify against the count we just sent in Step 5 (one prompt = one llm span).
this_run = len(prompts_a) + len(prompts_b)
target_spans = llm_spans[:this_run]

print(f'Total LLM spans returned: {len(llm_spans)}; verifying the most recent {len(target_spans)}.\n')
print(f'{"out_t":>6}  {"reasoning":>11}  {"expected":>11}  match')
print('-' * 50)

populated = matches = mismatches = 0
for s in target_spans:
    out_t = getattr(s.metrics, 'num_output_tokens', None) or 0
    ap = s.metrics.additional_properties or {}
    reasoning = ap.get('num_reasoning_tokens')
    expected = synthetic_reasoning_tokens(int(out_t))
    match = reasoning is not None and int(reasoning) == expected
    if reasoning is not None:
        populated += 1
        if match:
            matches += 1
        else:
            mismatches += 1
    print(f'{int(out_t):>6}  {str(int(reasoning)) if reasoning is not None else "None":>11}  {expected:>11}  {"YES" if match else "no"}')

n = len(target_spans)
print(f'\nReasoning tokens (synthetic, OTel path) on this run: populated={populated}/{n}; matches={matches}; mismatches={mismatches}')
if n > 0 and populated == n and matches == n and mismatches == 0:
    print(
        '\n  PROVEN (reasoning_tokens via OTel attribute):\n'
        '  Every LLM span\'s gen_ai.usage.details.reasoning_tokens attribute surfaced verbatim\n'
        '  as num_reasoning_tokens on the span metrics, matching the synthetic formula\n'
        '  out_t // 2 + 7 exactly. Path exercised:\n'
        '    span.set_attribute("gen_ai.usage.details.reasoning_tokens", N)\n'
        '      -> processor.py:65 (canonical attribute mapping)\n'
        '      -> processor.py:746,754 (writes metrics["num_reasoning_tokens"])\n'
        '      -> spans-search response.\n'
        '  The OTel processor does no transformation, no synthesis, no model-name check.'
    )

Total LLM spans returned: 6; verifying the most recent 6.

 out_t    reasoning     expected  match
--------------------------------------------------
    30           22           22  YES
    22           18           18  YES
    14           14           14  YES
    18           16           16  YES
    16           15           15  YES
    34           24           24  YES

Reasoning tokens (synthetic, OTel path) on this run: populated=6/6; matches=6; mismatches=0

  PROVEN (reasoning_tokens via OTel attribute):
  Every LLM span's gen_ai.usage.details.reasoning_tokens attribute surfaced verbatim
  as num_reasoning_tokens on the span metrics, matching the synthetic formula
  out_t // 2 + 7 exactly. Path exercised:
    span.set_attribute("gen_ai.usage.details.reasoning_tokens", N)
      -> processor.py:65 (canonical attribute mapping)
      -> processor.py:746,754 (writes metrics["num_reasoning_tokens"])
      -> spans-search response.
  The OTel processor does no transformation, no sy

## Step 7: Dashboard surfaces

In [10]:
print('=== UI surfaces ===')
print(f'Log stream       : {log_stream_url}')
print(f'Trends tab       : {trends_url}')
print(f'Settings/pricing : {urljoin(console_base, "settings/model-pricing")}')

=== UI surfaces ===
Log stream       : https://console-bin-citizens.gcp-dev.galileo.ai/project/a198016f-83d6-4d69-983f-391cb7c8ba8e/log-streams/5fa91e4f-ed10-442d-bbe8-97899bf30f38
Trends tab       : https://console-bin-citizens.gcp-dev.galileo.ai/project/a198016f-83d6-4d69-983f-391cb7c8ba8e/log-streams/5fa91e4f-ed10-442d-bbe8-97899bf30f38?tab=trends
Settings/pricing : https://console-bin-citizens.gcp-dev.galileo.ai/settings/model-pricing


## Step 8: Cleanup

Shut the OTel pipeline down cleanly so subsequent notebooks start fresh.

In [11]:
try:
    tracer_provider.shutdown()
except Exception:
    pass

# delete_project(name=PROJECT_NAME)
print(f'Project kept for inspection: {PROJECT_NAME}')
print(f'  Log stream : {log_stream_url}')

Project kept for inspection: GalileoEval_S17_CostOTel
  Log stream : https://console-bin-citizens.gcp-dev.galileo.ai/project/a198016f-83d6-4d69-983f-391cb7c8ba8e/log-streams/5fa91e4f-ed10-442d-bbe8-97899bf30f38
